# Full Decoder-Only Transformer — Solution

Reference implementation for `03_transformer_model_exercise.ipynb`. The capstone of the series.

## Setup

In [1]:
import torch 
import torch.nn as nn
import torch.nn.functional as F

In [2]:
def get_rope_frequencies(d_head, base=10000.0):
    return 1 / torch.pow(base, torch.arange(0, d_head, 2) / d_head)


def apply_rope(x, positions):
    """Apply RoPE to a Q or K tensor.

    Args:
        x:         (batch, seq_len, n_heads, d_head)
        positions: (seq_len,) integer positions

    Returns:
        Same shape as x, with each dim pair rotated.
    """
    x_rotated = torch.zeros(*x.shape)
    frequencies = get_rope_frequencies(x.shape[-1])         # (d_head//2,)
    angles = positions[:, None] * frequencies[None, :]      # (seq_len, d_head//2)

    cos = torch.cos(angles).unsqueeze(1)                    # (seq_len, 1, d_head//2)
    sin = torch.sin(angles).unsqueeze(1)

    x_rotated[..., 0::2] = x[..., 0::2] * cos - x[..., 1::2] * sin
    x_rotated[..., 1::2] = x[..., 0::2] * sin + x[..., 1::2] * cos
    return x_rotated

In [3]:
class SwiGLU(nn.Module):
    def __init__(self,d_model,d_ff=None):
        super().__init__()
        if d_ff is None:
            d_ff = int(8*d_model/3)

        self.w_gate = nn.Linear(d_model,d_ff,bias=False)
        self.w_up = nn.Linear(d_model,d_ff,bias=False)
        self.w_down = nn.Linear(d_ff,d_model,bias=False)

    def forward(self,x):
        gate = F.silu(self.w_gate(x))

        up_proj = self.w_up(x)

        out = gate * up_proj

        return self.w_down(out)

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, causal=False):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model//n_heads
        self.causal = causal

        self.qkv_projection = nn.Linear(d_model,d_model*3)
        self.out_projection = nn.Linear(d_model,d_model)

    def forward(self, x):
        # x -> bs,seq_len,d_model
        bs,seq_len,_ = x.shape 

        Q,K,V = torch.chunk(self.qkv_projection(x),3,dim=-1) # bs,seq_len,d_model

        Q = Q.view(bs,seq_len,self.n_heads,self.d_head) #bs,seq_len,n_head,d_head
        K = K.view(bs,seq_len,self.n_heads,self.d_head) #bs,seq_len,n_head,d_head
        V = V.view(bs,seq_len,self.n_heads,self.d_head) #bs,seq_len,n_head,d_head

        ## ROPE
        Q = apply_rope(Q, torch.arange(seq_len))
        K = apply_rope(K, torch.arange(seq_len))

        # Now permute for the attention matmul
        Q = Q.permute(0, 2, 1, 3)
        K = K.permute(0, 2, 1, 3)
        V = V.permute(0, 2, 1, 3)

        scores = Q@K.transpose(-2,-1) / self.d_head**0.5

        if self.causal:
            mask = torch.tril(torch.ones(seq_len,seq_len,dtype=torch.bool))
            scores = scores.masked_fill(~mask,float("-inf"))

        weights = torch.softmax(scores,dim=-1)
        out = weights@V

        out = out.permute(0,2,1,3).contiguous().view(bs,seq_len,self.d_model)
        out = self.out_projection(out)

        return out

## Solution 1 — `TransformerBlock`

In [5]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, causal=False):
        super().__init__()
        self.attention = MultiHeadAttention(
            d_model,
            n_heads,
            causal
        )

        self.norm_1 = nn.RMSNorm(d_model)
        self.norm_2 = nn.RMSNorm(d_model)

        self.ffn = SwiGLU(d_model)
    
    def forward(self,x):
        # x -> bs,seq_len,d_model
        attn_out = self.attention(self.norm_1(x))
        x = x + attn_out

        ffn_out = self.ffn(self.norm_2(x))
        x = x + ffn_out

        return x

## Solution 2 — `TransformerLM`

In [26]:
class TransformerLM(nn.Module):
    def __init__(self, 
                vocab_size,
                d_model, 
                n_heads, 
                n_layers, 
                causal=True
        ):
        super().__init__()

        self.embeddings = nn.Embedding(vocab_size,d_model)

        self.decoder_layers = nn.ModuleList([
            TransformerBlock(d_model,n_heads,causal) for _ in range(n_layers)
        ])

        self.norm = nn.RMSNorm(d_model)
        
        self.out_proj = nn.Linear(d_model,vocab_size)
    
    def forward(self,x):
        # x-> bs,seq_len
        x = self.embeddings(x)
        # x => bs,seq_len,d_model
        for layer in self.decoder_layers:
            x = layer(x)

        out = self.norm(x)
        out = self.out_proj(out) 

        return out

**Why pre-norm?** With post-norm, the residual connection passes through the normalization, which can attenuate gradients. With pre-norm, the residual is a clean additive highway from input to output — gradients flow back unchanged. This is what enables training 60+ layer transformers without warmup tricks.

**Why a final RMSNorm?** Pre-norm means the LAST sublayer's output goes directly into the residual stream without normalization. Without a final norm, the logits can have wildly varying scale across training. The final RMSNorm before the LM head ensures predictable input scale to the output projection.

**Why no biases?** With pre-norm, the normalization layer can absorb any constant shift. So biases become redundant — and removing them simplifies tensor parallelism (no need to handle bias replication across GPUs). Empirically, no quality loss.

## Solution 3 — Causality verification

In [7]:
vocab_size = 1000
d_model = 64
n_heads = 2 
n_layers = 2
causal= True


model = TransformerLM(
        vocab_size,
        d_model,
        n_heads,
        n_layers,
        causal
)

In [8]:
tokens_a = torch.randint(0, 1000, (1, 16))
tokens_b = tokens_a.clone()
tokens_b[0, 10:] = 999       # change positions 10..15
logits_a = model(tokens_a)
logits_b = model(tokens_b)
assert torch.allclose(logits_a[0, :10], logits_b[0, :10], atol=1e-5)

#   - This is the property that makes parallel teacher-forcing training valid

## Solution 4 — Greedy generation

In [23]:
def generate(model, prompt, max_new_tokens):
    #prompt - bs,seq_len (tokens)
    for _ in range(max_new_tokens):
        out = model(prompt) #bs,seq_len,d_model
        out = torch.argmax(out,axis=-1)[:,-1]
        prompt = torch.concat([prompt,out[None,:]],axis=-1)

    return prompt

In [24]:
generate(model,torch.randint(0,1000,(1,16)),10)

tensor([[462, 700, 551, 600, 652,  27, 260, 780, 772, 854, 310, 353, 947, 618,
         572, 990, 646,  75, 721,  35, 625, 199, 562, 562, 562, 562]])

**The honest gotcha.** _This `generate` function recomputes K, V for every previous token at every step — O(n²) total work for n tokens. Real LLMs use a KV cache to make each step O(1). That's the topic of `blog/05-attention-architectures/` (where MQA/GQA reduce the cache size) and is also why Flash Attention's prefill speedup matters so much. We'll skip the cache here for clarity._

## Run the tests

In [27]:
from tests import run_transformer_tests
run_transformer_tests(TransformerBlock, TransformerLM)

Running Transformer model...
  ✓ Block output shape correct
  ✓ Block gradients flow
  ✓ Model output shape correct
  ✓ Model gradients flow
  ✓ Causality holds end-to-end

  All 5 checks passed ✓



True